# CSIS3754 - Question 3: Medical Information Classification
## Main End-of-Year Examination 2024

## 3.1 - Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Load the dataset
med_info = pd.read_csv('med_info.csv')
print('Dataset loaded successfully!')
med_info.head()

## 3.2 - Probability of Female Patient with AB+ Blood Having Diabetes

In [ ]:
# Filter: Female AND AB+ blood type AND Diabetes medical condition
# P(Diabetes | Female AND AB+) = count(Female AND AB+ AND Diabetes) / count(Female AND AB+)

# Adjust column names below to match your actual dataset column names
# Common column names: 'Gender', 'Blood Type', 'Medical Condition'

female_abpos = med_info[
    (med_info['Gender'] == 'Female') &
    (med_info['Blood Type'] == 'AB+')
]

female_abpos_diabetes = female_abpos[
    female_abpos['Medical Condition'] == 'Diabetes'
]

prob = len(female_abpos_diabetes) / len(female_abpos)
print(f'Total Female AB+ patients:                {len(female_abpos)}')
print(f'Female AB+ patients with Diabetes:        {len(female_abpos_diabetes)}')
print(f'Probability (as percentage):              {round(prob * 100, 1)}%')

## 3.3 - Dates with Least Patients Admitted

In [ ]:
# Convert admission date column to datetime
# Adjust column name if needed (e.g. 'Date of Admission')
med_info['Date of Admission'] = pd.to_datetime(med_info['Date of Admission'], errors='coerce')

# Count admissions per date
admissions_per_date = med_info['Date of Admission'].value_counts()

# Find the minimum count and get all dates matching it
min_count = admissions_per_date.min()
least_admitted_dates = admissions_per_date[admissions_per_date == min_count]

print(f'Minimum number of admissions on any date: {min_count}')
print(f'\nDate(s) with the least patients admitted:')
print(least_admitted_dates)

## 3.4 - Correlation Matrix and Heatmap

In [ ]:
# To compute correlation we first need numeric columns.
# We temporarily encode all object columns for the correlation matrix.

med_encoded = med_info.copy()

for col in med_encoded.select_dtypes(include='object').columns:
    le = LabelEncoder()
    med_encoded[col] = le.fit_transform(med_encoded[col].astype(str))

# Drop date column if it is still datetime type
med_encoded = med_encoded.select_dtypes(include=[np.number])

corr_matrix = med_encoded.corr()

print('Correlation Matrix:')
print(corr_matrix)

In [ ]:
# Plot Heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    square=True,
    linewidths=0.5
)
plt.title('Correlation Heatmap - Medical Info Dataset', fontsize=13)
plt.tight_layout()
plt.show()

### 3.4.1 - Discussion of Correlation Matrix

In [ ]:
discussion_34 = """
INFERENCE FROM CORRELATION MATRIX AND HEATMAP:
===============================================

- Features with a correlation value close to +1 or -1 indicate a strong
  linear relationship. These features move together (positive) or in
  opposite directions (negative).

- Features with a correlation close to 0 have little to no linear
  relationship with each other.

- In medical datasets, features such as 'Age' and 'Billing Amount' often
  show moderate correlations with medical conditions and test results.

- Strong correlations between input features (multicollinearity) can
  negatively impact models like Logistic Regression. Highly correlated
  features may be candidates for removal to reduce redundancy.

- Examine which features correlate most strongly with 'Test Results'
  as these will be the most predictive features for our classifier.
"""
print(discussion_34)

## 3.5 - Remove Features with More than 10 Unique Text Values

In [ ]:
# Work on a copy of the original dataframe from here
med_clean = med_info.copy()

print('Unique text value counts per object column (before removal):')
for col in med_clean.select_dtypes(include='object').columns:
    print(f'  {col}: {med_clean[col].nunique()} unique values')

# Use a for loop to remove columns with more than 10 unique text values
cols_to_drop = []
for col in med_clean.select_dtypes(include='object').columns:
    if med_clean[col].nunique() > 10:
        cols_to_drop.append(col)

med_clean.drop(columns=cols_to_drop, inplace=True)

print(f'\nColumns removed (> 10 unique text values): {cols_to_drop}')
print(f'\nDataframe after removal:')
med_clean.head()

## 3.6 - Remove Insurance Provider, Billing Amount and Room Number

In [ ]:
# Remove specified columns (only if they exist)
cols_remove = ['Insurance Provider', 'Billing Amount', 'Room Number']
cols_remove_existing = [c for c in cols_remove if c in med_clean.columns]

med_clean.drop(columns=cols_remove_existing, inplace=True)

print(f'Removed columns: {cols_remove_existing}')
print(f'\nDataframe shape: {med_clean.shape}')
med_clean.head()

## 3.7 - Convert Text Values to Numeric

In [ ]:
# Step 1: Convert 'Test Results' (target label) using LabelEncoder
le_target = LabelEncoder()
med_clean['Test Results'] = le_target.fit_transform(med_clean['Test Results'])
print(f"'Test Results' classes: {list(le_target.classes_)} -> {list(range(len(le_target.classes_)))}")

med_clean.head()

In [ ]:
# Step 2: Binary encode features with exactly 2 unique text values (e.g. Gender: Male/Female)
object_cols = med_clean.select_dtypes(include='object').columns.tolist()

binary_cols = [col for col in object_cols if med_clean[col].nunique() == 2]
print(f'Binary columns (2 unique values): {binary_cols}')

le = LabelEncoder()
for col in binary_cols:
    med_clean[col] = le.fit_transform(med_clean[col])
    print(f"  '{col}' encoded.")

med_clean.head()

In [ ]:
# Step 3: One-hot encode remaining object columns with more than 2 unique values
# (excluding Test Results which is already encoded)
remaining_obj = med_clean.select_dtypes(include='object').columns.tolist()
print(f'Columns to one-hot encode: {remaining_obj}')

if remaining_obj:
    med_clean = pd.get_dummies(med_clean, columns=remaining_obj, drop_first=False)
    print('One-hot encoding applied.')

print(f'\nDataframe shape after encoding: {med_clean.shape}')
med_clean.head()

## 3.8 - Define X, y and Train/Test Split

In [ ]:
# Drop any remaining date columns (non-numeric)
non_numeric = med_clean.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric:
    med_clean.drop(columns=non_numeric, inplace=True)
    print(f'Dropped non-numeric columns: {non_numeric}')

# Define X (features) and y (target)
X = med_clean.drop(columns=['Test Results'])
y = med_clean['Test Results']

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

In [ ]:
# Train/Test split: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'X_train dimensions: {X_train.shape}')
print(f'y_train dimensions: {y_train.shape}')
print(f'X_test dimensions:  {X_test.shape}')
print(f'y_test dimensions:  {y_test.shape}')

In [ ]:
# Scale features - important for SVM and Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('Feature scaling applied (StandardScaler).')
print(f'X_train_scaled shape: {X_train_scaled.shape}')
print(f'X_test_scaled shape:  {X_test_scaled.shape}')

## 3.9 - Train Classifiers and K-Fold Cross-Validation

In [ ]:
# Define classifiers
classifiers = {
    'Naive Bayes':          GaussianNB(),
    'Logistic Regression':  LogisticRegression(),
    'SVM':                  SVC(gamma='auto')
}

k = 10
results_summary = {}

print(f'K-Fold Cross-Validation (k={k}) Results:')
print('='*60)

for name, clf in classifiers.items():
    # Cross-validated accuracy
    acc_scores  = cross_val_score(clf, X_train_scaled, y_train, cv=k, scoring='accuracy')
    # Cross-validated F1 (weighted for multi-class)
    f1_scores   = cross_val_score(clf, X_train_scaled, y_train, cv=k, scoring='f1_weighted')

    results_summary[name] = {
        'Mean Accuracy': acc_scores.mean(),
        'Std Accuracy':  acc_scores.std(),
        'Mean F1':       f1_scores.mean(),
        'Std F1':        f1_scores.std()
    }

    print(f'\n{name}:')
    print(f'  Training Accuracy:  {acc_scores.mean():.4f} (+/- {acc_scores.std():.4f})')
    print(f'  F1 Score:           {f1_scores.mean():.4f} (+/- {f1_scores.std():.4f})')

print('\n' + '='*60)

In [ ]:
# Learning Curves for each classifier
def plot_learning_curve(estimator, title, X, y, cv=10):
    train_sizes, train_scores, val_scores = learning_curve(
        estimator, X, y, cv=cv,
        train_sizes=np.linspace(0.1, 1.0, 10),
        scoring='accuracy', n_jobs=-1
    )
    train_mean = np.mean(train_scores, axis=1)
    train_std  = np.std(train_scores,  axis=1)
    val_mean   = np.mean(val_scores,   axis=1)
    val_std    = np.std(val_scores,    axis=1)

    plt.figure(figsize=(8, 5))
    plt.plot(train_sizes, train_mean, 'o--', color='blue',  label='Training Score')
    plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
    plt.plot(train_sizes, val_mean,   'o-',  color='green', label='Cross-Validation Score')
    plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color='green')
    plt.title(f'Learning Curve - {title}', fontsize=12)
    plt.xlabel('Training Set Size')
    plt.ylabel('Accuracy Score')
    plt.legend(loc='best')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

for name, clf in classifiers.items():
    plot_learning_curve(clf, name, X_train_scaled, y_train, cv=k)

## 3.10 - Best Model: Predictions and Evaluation

In [ ]:
# Select model with highest F1 score
best_model_name = max(results_summary, key=lambda k: results_summary[k]['Mean F1'])
best_model = classifiers[best_model_name]

print(f'Best model (highest F1): {best_model_name}')
print(f"  Mean F1 Score: {results_summary[best_model_name]['Mean F1']:.4f}")

In [ ]:
# Fit the best model on the full training set
best_model.fit(X_train_scaled, y_train)

# Predict on test set
y_pred = best_model.predict(X_test_scaled)

# Test Accuracy
test_acc = accuracy_score(y_test, y_pred)
print(f'Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)')

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print('Confusion Matrix:')
print(cm)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_target.classes_,
            yticklabels=le_target.classes_)
plt.title(f'Confusion Matrix - {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# Classification Report
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=le_target.classes_))

## 3.11 - Inference from Metrics and Learning Curves

In [ ]:
discussion_311 = """
INFERENCE FROM LEARNING CURVES, METRICS AND SCORES:
====================================================

Learning Curves:
- If the training score is much higher than the cross-validation score and
  the gap does not close as training size increases -> OVERFITTING.
  The model is memorising training data and not generalising well.

- If both training and CV scores are low and plateau at similar values
  -> UNDERFITTING. The model is too simple to capture the underlying patterns.

- If the training and CV scores converge at a reasonably high value
  -> GOOD FIT. The model generalises well.

Metrics (Accuracy, F1, Confusion Matrix):
- The F1 score balances precision and recall, making it more informative
  than accuracy alone for imbalanced datasets.
- The confusion matrix shows which classes are misclassified most often.
- If a particular class has low recall, the model struggles to correctly
  identify that class.

Recommendations:
1. If overfitting: Apply regularisation (e.g. increase C in SVM/LR),
   reduce model complexity, or gather more training data.
2. If underfitting: Use a more complex model, add more informative features,
   or reduce regularisation.
3. If class imbalance is present: Apply oversampling (SMOTE) or use
   class_weight='balanced' parameter.
4. Consider feature engineering or selection to improve model performance.
"""
print(discussion_311)